In [8]:
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

polygons_path = r"C:\Users\KevinWilliams\Documents\Source Files\CVG ROW Solar\Exports\BasePolys.shp"
nlcd_path = r"C:\Users\KevinWilliams\Documents\Source Files\CVG ROW Solar\Rasters\NLCD_CONUS_5070_clipped.tif"
out_csv = r"C:\Users\KevinWilliams\Documents\Source Files\CVG ROW Solar\NLCD\NLCD.csv"

# Load your shapefile
polygons = gpd.read_file(polygons_path)

# Load raster to check its CRS
with rasterio.open(nlcd_path) as src:
    raster_crs = src.crs

# Reproject polygons to match raster CRS (critical - NLCD is often in Albers Equal Area)
if polygons.crs != raster_crs:
    polygons = polygons.to_crs(raster_crs)

# Compute zonal stats - "majority" = mode, "count" = pixel count used
stats = zonal_stats(
    polygons,
    nlcd_path,
    stats=["majority", "count"],
    nodata=None,       # set this to your raster's actual nodata value if it has one
    categorical=False,  # not needed since we're using majority, not full histogram
    all_touched=True
)

# Attach results back to the GeoDataFrame
polygons["mode_value"] = [s["majority"] for s in stats]
polygons["pixel_count"] = [s["count"] for s in stats]

print(polygons[["mode_value", "pixel_count"]])

   mode_value  pixel_count
0        21.0           38
1        23.0            8
2        23.0           10
3        23.0            7
4        22.0           11
5        21.0           39
